In [ ]:
# =========================
# IMPORTS
# =========================
import os
import cv2
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import Xception, EfficientNetB0
from tensorflow.keras.applications.xception import preprocess_input as x_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as e_preprocess
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# =========================
# CONFIG
# =========================
IMG_SIZE = 224
DATASET_PATH = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake"

# =========================
# LOAD DATASET
# =========================
def load_dataset(data_dir):
    images, labels = [], []

    for label, category in enumerate(["real", "fake"]):
        path = os.path.join(data_dir, category)

        for file in os.listdir(path):
            img_path = os.path.join(path, file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            labels.append(label)

    return np.array(images), np.array(labels)

# =========================
# BUILD MODEL
# =========================
def build_model(base_model):
    x = base_model.output
    x = Dense(256, activation='relu')(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=output)

    for layer in base_model.layers:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# =========================
# TRAIN MODELS
# =========================
def train_models():
    X, y = load_dataset(DATASET_PATH)

    # Preprocess separately
    X_x = x_preprocess(X.copy())
    X_e = e_preprocess(X.copy())

    # Base models
    x_base = Xception(weights="imagenet", include_top=False, pooling="avg")
    e_base = EfficientNetB0(weights="imagenet", include_top=False, pooling="avg")

    # Build
    x_model = build_model(x_base)
    e_model = build_model(e_base)

    print("Training Xception...")
    x_model.fit(X_x, y, epochs=5, batch_size=16, validation_split=0.2)

    print("Training EfficientNet...")
    e_model.fit(X_e, y, epochs=5, batch_size=16, validation_split=0.2)

    # Save
    x_model.save("/kaggle/working/xception_model.h5")
    e_model.save("/kaggle/working/efficient_model.h5")

    return x_model, e_model

# =========================
# LOAD MODELS
# =========================
def load_models():
    x_model = tf.keras.models.load_model("/kaggle/working/xception_model.h5")
    e_model = tf.keras.models.load_model("/kaggle/working/efficient_model.h5")

    return {"xception": x_model, "efficient": e_model}

# =========================
# MULTI-VIEW
# =========================
def generate_views(image):
    return [
        image,
        cv2.flip(image, 1),
        cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE),
        cv2.GaussianBlur(image, (5,5), 0)
    ]

# =========================
# PREPROCESS
# =========================
def preprocess(image, model_type):
    image = cv2.resize(image, (224, 224))
    image = np.expand_dims(image, axis=0)

    if model_type == "xception":
        return x_preprocess(image)
    else:
        return e_preprocess(image)

# =========================
# PREDICTIONS
# =========================
def get_predictions(models, views):
    x_preds, e_preds = [], []

    for view in views:
        x_input = preprocess(view, "xception")
        e_input = preprocess(view, "efficient")

        x_preds.append(models["xception"].predict(x_input, verbose=0)[0][0])
        e_preds.append(models["efficient"].predict(e_input, verbose=0)[0][0])

    return x_preds, e_preds

# =========================
# AGGREGATION
# =========================
def aggregate(preds):
    return float(np.mean(preds))

# =========================
# WEIGHTED FUSION
# =========================
def weighted_fusion(x, e, w1=0.6, w2=0.4):
    return w1 * x + w2 * e

# =========================
# FUZZY LOGIC
# =========================
def fuzzy_decision(x, e):
    def high(v): return max(0, min(1, (v - 0.5) * 2))

    x_h = high(x)
    e_h = high(e)

    rule1 = min(x_h, e_h)
    rule2 = max(x_h, e_h)

    return 0.7 * rule1 + 0.3 * rule2

# =========================
# FINAL PREDICTION
# =========================
def predict(image_path, models):
    image = cv2.imread(image_path)

    views = generate_views(image)

    x_preds, e_preds = get_predictions(models, views)

    x_score = aggregate(x_preds)
    e_score = aggregate(e_preds)

    fused = weighted_fusion(x_score, e_score)
    fuzzy = fuzzy_decision(x_score, e_score)

    result = "FAKE" if fuzzy > 0.5 else "REAL"

    return result, {
        "xception": x_score,
        "efficient": e_score,
        "fused": fused,
        "fuzzy": fuzzy
    }

# =========================
# RUN
# =========================
if __name__ == "__main__":

    # LOAD DATA
    X_train, y_train = load_dataset(DATASET_PATH, "train")
    X_val, y_val = load_dataset(DATASET_PATH, "valid")
    X_test, y_test = load_dataset(DATASET_PATH, "test")

    # PREPROCESS
    X_train_x = x_preprocess(X_train.copy())
    X_val_x = x_preprocess(X_val.copy())

    X_train_e = e_preprocess(X_train.copy())
    X_val_e = e_preprocess(X_val.copy())

    # BUILD MODELS
    x_base = Xception(weights="imagenet", include_top=False, pooling="avg")
    e_base = EfficientNetB0(weights="imagenet", include_top=False, pooling="avg")

    x_model = build_model(x_base)
    e_model = build_model(e_base)

    # TRAIN
    print("Training Xception...")
    x_model.fit(X_train_x, y_train, validation_data=(X_val_x, y_val), epochs=5, batch_size=16)

    print("Training EfficientNet...")
    e_model.fit(X_train_e, y_train, validation_data=(X_val_e, y_val), epochs=5, batch_size=16)

    # SAVE MODELS
    x_model.save("/kaggle/working/xception_model.h5")
    e_model.save("/kaggle/working/efficient_model.h5")

    models = {"xception": x_model, "efficient": e_model}

    # TEST EVALUATION
    X_test_x = x_preprocess(X_test.copy())
    X_test_e = e_preprocess(X_test.copy())

    x_preds = x_model.predict(X_test_x).flatten()
    e_preds = e_model.predict(X_test_e).flatten()

    final_preds = (x_preds + e_preds) / 2
    y_pred_labels = (final_preds > 0.5).astype(int)

    from sklearn.metrics import classification_report, confusion_matrix

    print("\n=== TEST RESULTS ===")
    print(confusion_matrix(y_test, y_pred_labels))
    print(classification_report(y_test, y_pred_labels))
